# ENVIRONMENT SETUP & DEPENDENCY CONFIGURATION
Objective: Prepare the cloud virtual machine (Google Colab) environment with the necessary web automation tools and data libraries required to extract real-time voter registration data from the official IEC portal.

In [ ]:
# Install required libraries and Chromium browser tools for Colab
!apt-get update!
!pip install webdriver-manager selenium pandas
!apt-get install -y chromium-browser chromium-chromedriver
!pip install -q playwright
!pip install chromium
!playwright install chromium

# Install Playwright Dependencies
!playwright install-deps

# !apt-get remove -y chromium-browser chromium-chromedriver

# HEADLESS BROWSER INITIALIZATION & CONNECTION TEST

Objective: Configure Selenium WebDriver with headless Chrome options optimized for cloud execution environments (Google Colab). This block handles binary path  mapping and performs a live connectivity test against the official Electoral Commission of South Africa (IEC) Voter Registration Statistics portal.

Importing Data From IEC database & Statistics they have collected.

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import time
import pandas as pd

options = Options()
options.add_argument('--headless=new')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-gpu')
options.add_argument("window-size=1920,1080")

# Point directly to the system-installed binary paths on Colab
service = Service('/usr/bin/chromedriver')
driver = webdriver.Chrome(service=service, options=options)

try:
    url = "https://www.elections.org.za/pw/StatsData/Voter-Registration-Statistics"
    driver.get(url)
    time.sleep(5)
    print("SUCCESS! Page title:", driver.title)

finally:
    driver.quit()

# HTTP CONNECTIVITY & API PROBING TEST

Objective: Test whether the target Electoral Commission of South Africa (IEC) statistical page is accessible via a lightweight HTTP GET request, and  determine if the page content is statically rendered (parseable via BeautifulSoup) or dynamically loaded via client-side JavaScript (requiring Selenium).

In [ ]:
import requests

# Test a direct request to see if the page responds or exposes an API
url = "https://www.elections.org.za/pw/StatsData/Voter-Registration-Statistics"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
print("Response error Code:", response.status_code)

if response.status_code == 200:
    print("Page is reachable")
else:
    print("check issues")

# ASYNCHRONOUS HEADLESS BROWSER AUTOMATION (PLAYWRIGHT)
---to delete

Objective: Initialize an asynchronous headless browser session using Playwright to navigate the dynamic Electoral Commission of South Africa (IEC) voter  registration portal. Playwright is selected over synchronous alternatives to efficiently handle asynchronous DOM rendering and dynamic dropdown APIs across multiple provincial endpoints.

# **Testing Setup to access the IEC Data**


# **Acccessing Data in the filters (i.e by Province)**

In [ ]:
import asyncio
from playwright.async_api import async_playwright
import pandas as pd

async def extract_iec_data():
    async with async_playwright() as p:
        # Launch browser in headless mode
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        # Navigate to the dashboard
        print("Navigating to IEC Voter Registration portal...")
        await page.goto("https://www.elections.org.za/pw/StatsData/Voter-Registration-Statistics")

        # Wait for the main container or dropdowns to load
        await page.wait_for_selector("select", timeout=10000)
        print("Page loaded successfully.")

        # Find all select dropdowns on the page
        dropdowns = await page.locator("select").all()
        print(f"Found {len(dropdowns)} dropdown menus on the page.")

        # Example: Extracting province options from the first dropdown
        province_select = dropdowns[0]
        province_options = await province_select.locator("option").all_inner_texts()
        print("Provinces available:", province_options)

        # You can loop through provinces, select them, and extract the resulting data cards/tables
        extracted_data = []

        for province in province_options[1:]: # Skip "All provinces" option placeholder
            print(f"Selecting province: {province}")
            await province_select.select_option(label=province)

            # Wait a moment for dynamic content/municipality dropdown to update
            await page.wait_for_timeout(3000)

            # Grab summary text or demographic numbers displayed on the page
            # (Adjust selectors based on specific text elements you want to target)
            summary_cards = await page.locator(".col-md-6, .card, table").all_inner_texts()

            extracted_data.append({
                "Province": province,
                "Raw_Content": summary_cards[:3] # Capture the top summary text blocks
            })

        await browser.close()

        # Convert to DataFrame and save
        df = pd.DataFrame(extracted_data)
        df.to_csv("iec_extracted_snapshot.csv", index=False)
        print("Extraction complete! Saved to 'iec_extracted_snapshot.csv'.")

# Run the async scraper function in Colab
await extract_iec_data()

# ***MUNICIPALITIES IN PROVINCES...  DATA EXTRACTION PIPELINE***

It opens the voter registration statistics page, then works through the province dropdown one at a time. For each province, it waits for the municipality dropdown to refresh, then loops through every municipality in that province.

In [ ]:
import asyncio
from playwright.async_api import async_playwright
import pandas as pd

async def extract_clean_iec_data():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        print("Opening IEC portal...")
        await page.goto("https://www.elections.org.za/pw/StatsData/Voter-Registration-Statistics")
        await page.wait_for_selector("select", timeout=15000)

        # Get province dropdown options
        province_options = await page.locator("select").first.locator("option").all_inner_texts()
        provinces = [p.strip() for p in province_options if p.strip() and p != "All provinces"]

        structured_data = []

        for province in provinces:
            print(f"Processing province: {province}")
            # Select province
            await page.locator("select").first.select_option(label=province)
            await page.wait_for_timeout(2000) # Wait for municipalities to update

            # Get municipality dropdown options for this province
            muni_dropdown = page.locator("select").nth(1)
            muni_options = await muni_dropdown.locator("option").all_inner_texts()
            municipalities = [m.strip() for m in muni_options if m.strip() and m != "All municipalities"]

            for muni in municipalities:
                print(f"  -> Selecting municipality: {muni}")
                await muni_dropdown.select_option(label=muni)
                await page.wait_for_timeout(2500) # Wait for demographic cards to load

                # Extract demographic statistics blocks (target specific cards/tables containing age groups)
                # We look for text blocks containing numbers and age brackets
                stats_elements = await page.locator(".col-md-6, .card-body, .row").all_inner_texts()

                for text in stats_elements:
                    if "18 - 19" in text or "20 - 29" in text: # Look for the demographic panel
                        structured_data.append({
                            "Province": province,
                            "Municipality": muni,
                            "Raw_Demographics": text.strip()
                        })

        await browser.close()

        # Save clean results
        df_clean = pd.DataFrame(structured_data)
        df_clean.to_csv("iec_clean_demographics.csv", index=False)
        print("Clean extraction complete! Saved to 'iec_clean_demographics.csv'.")

# Run it
await extract_clean_iec_data()

# REGEX-BASED PARSING & DEMOGRAPHIC STRUCTURING PIPELINE
Objective: Transform unstructured raw text payloads (scraped from the IEC portal) into a clean, tabular, analysis-ready dataset.

-----
This script leverages regular expressions (regex) to systematically extract total registered voters, overall gender distributions, and cohort-specific age brackets (from 18–19 to 80+), computing composite sub-totals and persisting the schema to disk.

In [ ]:
import pandas as pd
import re

# 1. Load your raw demographic text dataset
df_raw = pd.read_csv('iec_clean_demographics.csv')

parsed_data = []
age_groups = ['18 - 19', '20 - 29', '30 - 39', '40 - 49', '50 - 59', '60 - 69', '70 - 79', '80+']

for idx, row in df_raw.iterrows():
    prov = row['Province']
    muni = row['Municipality']
    text = row['Raw_Demographics']

    # Initialize dictionary for the row
    row_dict = {
        'Province': prov,
        'Municipality': muni
    }

    # Extract total registered voters
    total_match = re.search(r'Total\n([0-9,]+)', text)
    row_dict['Total_Registered_Voters'] = int(total_match.group(1).replace(',', '')) if total_match else None

    # Extract overall male and female counts
    overall_male = re.search(r'Overall\nMale\n[^\n]+\n([0-9,]+)', text)
    overall_female = re.search(r'Female\n[^\n]+\n([0-9,]+)', text)

    row_dict['Total_Male'] = int(overall_male.group(1).replace(',', '')) if overall_male else None

    # Extract age-group specific counts (Male & Female)
    for ag in age_groups:
        escaped_ag = re.escape(ag)
        pattern = rf'{escaped_ag}\nMale\n[^\n]+\n([0-9,]+)\nFemale\n[^\n]+\n([0-9,]+)'
        match = re.search(pattern, text)

        if match:
            male_val = int(match.group(1).replace(',', ''))
            female_val = int(match.group(2).replace(',', ''))
            row_dict[f'Male_{ag}'] = male_val
            row_dict[f'Female_{ag}'] = female_val
            row_dict[f'Total_{ag}'] = male_val + female_val
        else:
            row_dict[f'Male_{ag}'] = None
            row_dict[f'Female_{ag}'] = None
            row_dict[f'Total_{ag}'] = None

    parsed_data.append(row_dict)

# Convert to a clean structured DataFrame
df_cleaned = pd.DataFrame(parsed_data)

# Export the cleaned structured dataset to a new CSV file
df_cleaned.to_csv('iec_fully_cleaned_demographics.csv', index=False)

print("Cleaning and parsing complete!")
print(f"Total municipalities processed: {len(df_cleaned)}")
print(df_cleaned[['Province', 'Municipality', 'Total_Registered_Voters', 'Total_18 - 19', 'Total_20 - 29']].head())

# WARD-LEVEL GRANULARITY PARSING & ELECTORAL DATA EXTRACTION
Objective: Disaggregate municipal-level scraped payloads down to the granular ward level.

Here we were isolating official Ward IDs, voting district counts, and registered voter totals, transforming unstructured hierarchical data into a structured spatial panel
`saved as 'iec_registered_people_per_ward.csv'.`

In [ ]:
import pandas as pd
import re

# Load the raw scraped dataset we already saved
df_raw = pd.read_csv('iec_clean_demographics.csv')

ward_rows = []

for idx, row in df_raw.iterrows():
    prov = row['Province']
    muni = row['Municipality']
    text = row['Raw_Demographics']

    # Check if the ward-level table exists in the text block
    if "Registered voters count by province by municipality by ward" in text:
        parts = text.split("Registered voters count by province by municipality by ward")
        ward_section = parts[1]

        # Regex to capture Ward ID, Voting Districts count, and Registered Voters count
        matches = re.findall(r'([0-9]{8})\t([0-9]+)\t([0-9,]+)', ward_section)

        for ward_id, vd_count, reg_voters in matches:
            ward_rows.append({
                'province': prov,
                'municipality': muni,
                'ward_id': ward_id,
                'voting_districts': int(vd_count),
                'registered_people': int(reg_voters.replace(',', ''))
            })

# Convert to DataFrame
df_wards = pd.DataFrame(ward_rows)

# Export to a clean CSV file
df_wards.to_csv('iec_registered_people_per_ward.csv', index=False)

print(f"Successfully extracted {len(df_wards)} wards across all municipalities!")
print(df_wards.head())